# Train RecurrentSurrogate on multilayer spectra

An example training workflow using the current iBeamLab API.
Each layer has **one elemental areal-density input per element**, with no extra
total-thickness input: $N_e = N_\mathrm{layer}c_e$. Physical AD is in
$10^{15}$ atoms/cm² and is multiplied once by `ad_input_factor`.

One shared RecurrentSurrogate learns all selected layer counts; each detector has its own
spectrum head and output processing. Preparation streams completed datasets
into disk-backed arrays. Training and evaluation read bounded minibatches.

Use a Python kernel with `torch`, `numpy`, `tqdm`, `matplotlib`, `onnx` and
`onnxruntime` installed. Notebook support requires `jupyterlab` or `ipykernel`.
Choose paths and settings below before running the remaining cells.

In [ ]:
from pathlib import Path
import json
import sys

# Locate the checkout from either the repository root or this notebook's folder.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "examples/ibeamnet/surrogate.toml").is_file()
             and (p / "ibeamlab").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the iBeamLab checkout")
sys.path.insert(0, str(ROOT))

# Change to an ONNX-enabled build for native deployment verification.
# Set None to use the installed extension instead.
EXTENSION_DIR = ROOT / "build/local/Release/bin"
if EXTENSION_DIR is not None:
    if not EXTENSION_DIR.is_dir():
        raise FileNotFoundError(EXTENSION_DIR)
    sys.path.insert(0, str(EXTENSION_DIR))

import numpy as np
import matplotlib.pyplot as plt
import torch
import ibeamlab as ibl
from examples.ibeamnet.configuration import load_config
from examples.ibeamnet.validation import validate_training, validate_checkpoint_contract
from examples.ibeamnet.workflow import load_cache
from examples.ibeamnet.data import digest
from examples.ibeamnet.training import train, make_model, load_checkpoint
from examples.ibeamnet.deployment import export

print("Repository:", ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())

## 1. Dataset, preprocessing and run settings

Paths in `surrogate.toml` resolve relative to that file. Override them below as
needed. Select only completed `layers_XX` datasets. A limit of 0 uses every
successful record; try 1,000 per layer system for an initial experiment.

Cache/output directories must be new, except when reusing a matching completed
cache or resuming an existing run. With 2.66 million 5,000-channel spectra,
cached targets alone require about 53.2 GB of disk space. Avoid changing AD or
output scaling after preparing a cache; choose a new cache path instead.

In [ ]:
CONFIG_PATH = ROOT / "examples/ibeamnet/surrogate.toml"
config = load_config(CONFIG_PATH)

# Optional weights-only initialization. It selects the saved architecture,
# ordered spectrum schema and AD scaling before preparing the training cache.
INITIAL_WEIGHTS = None
# INITIAL_WEIGHTS = ROOT / "examples/ibeamnet/checkpoints/reference_rbs/best.pt"
if INITIAL_WEIGHTS is not None:
    initial = load_checkpoint(INITIAL_WEIGHTS)
    config["model"] = initial["model_options"]
    config["spectra"] = initial["spectra"]
    config["preprocessing"]["ad_input_factor"] = initial["ad_input_factor"]
    config["data"]["cache"] = ROOT / "examples/ibeamnet" / ("cache-initial-" + digest(Path(INITIAL_WEIGHTS))[:12])
    print("Initial weights:", INITIAL_WEIGHTS)
    print("Initial weight elements:", initial["elements"])

# Dataset and cache settings. Uncomment to customize.
# config["data"]["source"] = ROOT / "examples/DataGeneration/datasets/iba_multilayer_13el"
# config["data"]["layer_counts"] = [1, 2]
# config["data"]["max_records_per_source"] = 1000
# config["data"]["cache"] = ROOT / "examples/ibeamnet/cache-small"
# config["preprocessing"]["ad_input_factor"] = 1e-4

# The schedule defines the TOTAL run, including epochs already completed.
RESUME_TRAINING = False
TRAINING_SCHEDULE = [
    {"learning_rate": 5e-4, "epochs": 25},
    {"learning_rate": 1e-4, "epochs": 90},
    {"learning_rate": 1e-5, "epochs": 50},
]
config["training"]["schedule"] = TRAINING_SCHEDULE
config["training"]["workers"] = 0  # Safe for Windows/Jupyter.
# config["training"]["output"] = ROOT / "examples/ibeamnet/runs/experiment-01"
# config["training"]["batch_size"] = 512
# config["training"]["device"] = "auto"  # auto, cpu, cuda, cuda:0
# config["model"]["hidden_size"] = 256

# Export is optional. Native verification needs an ONNX-enabled C++ extension.
RUN_EXPORT = False
VERIFY_NATIVE = True
# config["export"]["output"] = ROOT / "examples/ibeamnet/packages/experiment-01"

TOTAL_EPOCHS = validate_training(config["training"])
CACHE = Path(config["data"]["cache"])
RUN = Path(config["training"]["output"])
for key in ("source", "cache", "layer_counts", "max_records_per_source"):
    print(f"{key}: {config['data'][key]}")
print("AD input factor:", config["preprocessing"]["ad_input_factor"])
print("Detector outputs:", config["spectra"])
print("Run:", RUN)
print(f"Training schedule: {TOTAL_EPOCHS} total epochs, batch size {config['training']['batch_size']}")
start = 1
for stage, settings in enumerate(TRAINING_SCHEDULE, 1):
    stop = start + settings["epochs"] - 1
    print(f"Stage {stage}: epochs {start}..{stop}, learning rate {settings['learning_rate']:g}")
    start = stop + 1

## 2. Prepare or reuse a completed cache

Input arrays have shape `[samples, maximum_layers, elements]`; absent layers
are zero-padded. Spectra are padded independently to each configured width;
nonzero tails are rejected. Train/validation/test splits are seeded per layer
count, with all detector spectra from one sample kept in the same split.

An incomplete cache is never reused. Preparation does not modify source datasets.

In [ ]:
manifest = load_cache(config, prepare_if_missing=True)

print("Records:", f"{manifest['records']:,}")
print("Splits:", manifest["split_sizes"])
print("Elements:", manifest["elements"])
print("Network inputs per layer:", len(manifest["elements"]))
print("Input representation:", manifest["input_representation"])

## 3. Inspect spectra and areal-density distributions

Only a bounded selection of training rows is copied for plots. Counts are
shown before and after scaling; input histograms show physical elemental AD
summed over the sample's layers. Padding remains exactly zero.

In [ ]:
features = np.load(CACHE / "features.npy", mmap_mode="r")
targets = np.load(CACHE / "targets.npy", mmap_mode="r")
train_indices = np.load(CACHE / "train_indices.npy", mmap_mode="r")
representative = int(train_indices[len(train_indices) // 2])
fig, axes = plt.subplots(len(manifest["spectra"]), 2, figsize=(13, 4 * len(manifest["spectra"])), squeeze=False)
offset = 0
for row, spec in enumerate(manifest["spectra"]):
    stop = offset + spec["channels"]
    scaled = np.array(targets[representative, offset:stop])
    axes[row, 0].plot(scaled * spec["count_scale"])
    axes[row, 0].set(title=f"{spec['measurement']}: physical counts", xlabel="Channel", ylabel="Counts")
    axes[row, 1].plot(scaled)
    axes[row, 1].set(title=f"Counts / {spec['count_scale']:g}", xlabel="Channel", ylabel="Scaled counts")
    offset = stop
fig.tight_layout()
plt.show()

positions = np.linspace(0, len(train_indices) - 1, min(4096, len(train_indices)), dtype=np.int64)
totals = features[train_indices[positions]].sum(axis=1) / manifest["ad_input_factor"]
columns = 4
rows = (len(manifest["elements"]) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(14, 3 * rows), squeeze=False)
for index, element in enumerate(manifest["elements"]):
    axes.flat[index].hist(totals[:, index], bins=40)
    axes.flat[index].set(title=element, xlabel="Sample AD (1e15 atoms/cm²)", ylabel="Samples")
for axis in axes.flat[len(manifest["elements"]):]:
    axis.set_visible(False)
fig.tight_layout()
plt.show()
del totals

## 4. Inspect the recurrent surrogate and train

The shared GRU receives scaled elemental AD. Fixed-setup FiLM conditioning,
summed latent contributions and local spectrum refinement follow the reference.
Independent heads support different detector channel counts.

`TRAINING_SCHEDULE` above controls each stage's learning rate and epoch count.
Defaults are **25 epochs at 5e-4, 90 at 1e-4, then 50 at 1e-5**. All stages use
the same batch size. AdamW moments continue across boundaries; only its rate
changes. There is no automatic rate reduction or early stopping.

`best.pt` protects the best full-validation weights across the entire run,
including the initial baseline. `last.pt` stores the latest completed epoch,
AdamW state and random states. Both are saved before the first update too.

For continuation, set `RESUME_TRAINING=True`. The schedule is the TOTAL run,
not an additional block. After 65 completed epochs, a 165-epoch schedule runs
100 more. You may extend or revise the schedule; past history stays intact and
remaining epochs follow the revised stage boundaries. Other training settings
must match. An interrupted partial epoch is replayed from its saved start.
Held-out test data never drives rate changes or model selection.

No production training starts until you execute the training cell.

To fine-tune saved weights, set `INITIAL_WEIGHTS` above and use a new run
folder with `RESUME_TRAINING=False`. The source architecture, output widths,
scaling and element order must match. A new AdamW optimizer and schedule start
from epoch 1; the baseline is reevaluated on the current validation split.
`RESUME_TRAINING=True` restores this new run's optimizer and progress instead.
Keep `INITIAL_WEIGHTS` set so its input/output configuration remains selected.
The supplied reference checkpoint has 2,998 RBS channels; preparation rejects
nonzero spectral tails rather than truncating them.

In [ ]:
preview_model = make_model(manifest, config["model"])
print(preview_model)
print("Trainable parameters:", f"{sum(p.numel() for p in preview_model.parameters()):,}")
print("Architecture:", config["model"])
print("Training:", config["training"])
print("Loss:", config["loss"])
del preview_model

In [ ]:
training_result = train(config, resume=RESUME_TRAINING,
                        initial_weights=None if RESUME_TRAINING else INITIAL_WEIGHTS)
print("Best checkpoint:", training_result["checkpoint"])
print("Held-out test metrics:", training_result["test"])

## 5. Inspect validation history and held-out predictions

Reload the durable best checkpoint, not the final live weights. Plots use at
most eight test samples; full held-out metrics were streamed during training.
Relative L1 is reported independently for each detector in `test-metrics.json`.

In [ ]:
history = json.loads((RUN / "history.json").read_text())
metrics = json.loads((RUN / "test-metrics.json").read_text())
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
epochs = [h["epoch"] for h in history]
axes[0].plot(epochs, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs, [h["validation_loss"] for h in history], label="Validation")
axes[0].set(ylabel="Scaled-target loss", yscale="log")
axes[0].legend()
axes[1].step(epochs, [h["learning_rate"] for h in history], where="post", label="Actual learning rate")
axes[1].set(xlabel="Epoch", ylabel="Learning rate", yscale="log")
axes[1].legend()
for axis in axes:
    axis.grid(alpha=.25)
fig.tight_layout()
plt.show()
print(json.dumps(metrics, indent=2))

state = load_checkpoint(RUN / "best.pt")
validate_checkpoint_contract(state, manifest, cache_sha256=digest(CACHE / "manifest.json"))
best_model = make_model(manifest, state["model_options"])
best_model.load_state_dict(state["model_state"])
selected_device = config["training"]["device"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if selected_device == "auto" else torch.device(selected_device)
best_model.to(device).eval()
test_indices = np.load(CACHE / "test_indices.npy", mmap_mode="r")
selected = np.asarray(test_indices[:min(8, len(test_indices))])
with torch.inference_mode():
    predicted = best_model(torch.from_numpy(features[selected]).to(device)).cpu().numpy()
expected = np.array(targets[selected])
fig, axes = plt.subplots(len(selected), len(manifest["spectra"]),
    figsize=(7 * len(manifest["spectra"]), 2.8 * len(selected)), squeeze=False)
offset = 0
for detector, spec in enumerate(manifest["spectra"]):
    stop = offset + spec["channels"]
    for row, index in enumerate(selected):
        axis = axes[row, detector]
        axis.plot(expected[row, offset:stop] * spec["count_scale"], label="Simulation")
        axis.plot(predicted[row, offset:stop] * spec["count_scale"], "--", label="RecurrentSurrogate")
        source = next(s for s in manifest["sources"] if s["start"] <= index < s["start"] + s["records"])
        axis.set(title=f"{spec['measurement']}, {source['layer_count']} layers, cache row {index}", xlabel="Channel", ylabel="Counts")
        axis.grid(alpha=.25)
        if row == 0:
            axis.legend()
    offset = stop
fig.tight_layout()
plt.show()

## 6. Export one variable-layer model

Set `RUN_EXPORT=True` above to create one format-1 `forward.ibm` package.
The package has no fixed or maximum layer count. Its ONNX graph uses a runtime
Loop with dynamic batch/layer axes and skips trailing zero layers. Interior
zero layers remain masked no-ops. Training layer counts are provenance only.

The graph receives scaled elemental AD with one input per element per layer.
The C++ runtime converts physical thickness/fractions to AD, or accepts physical
AD directly with `model.predict_areal_density(ad)`. The AD factor is explicit
in `LayerInputSchema.preprocessing` and applies once outside the graph.
Each detector retains independent count scaling, optional exposure scaling and
rebinning. Exposure scaling requires pileup-free targets linear in particlesSr.

ONNX/PyTorch comparisons cover batch sizes 1 and 3 and multiple layer counts,
including counts beyond training. Native verification additionally checks the
C++ inference path. For a SIMNRA-only extension, set `VERIFY_NATIVE=False`;
deployment later requires an ONNX-enabled build. Restart the kernel before
switching compiled extension builds. Existing package directories are never
overwritten.

In [ ]:
if RUN_EXPORT:
    config["export"]["verify_native"] = VERIFY_NATIVE
    reports = export(config, checkpoint=RUN / "best.pt")
    print(json.dumps(reports, indent=2))
    print("Packages:", config["export"]["output"])
else:
    print("Export skipped. Set RUN_EXPORT=True to export the durable best checkpoint.")